# Analyse event counts without donor labels

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sthsci/Orca/blob/main/notebooks/04_event_count_analysis.ipynb)

**Analysis notebook.** Upload one to four experimental conditions, validate the public ORCA schema, explore the counts, fit population models, and export a reproducible result archive.

Run the cells from top to bottom. Values collected near the start of each notebook are safe places to experiment. Bayesian SMC fitting is deliberately disabled by default in the analysis notebooks because it can take several minutes; set `RUN_INFERENCE = True` when the data checks and descriptive plots look right.

Use synthetic or approved anonymised data only. Do not upload names, clinical metadata, raw microscopy, or a donor key that could identify participants.


## Input schema

One row represents one cell and one count outcome (for example total contacts or kills).

```csv
cell_id,condition,count
cell_001,Control,3
cell_002,Control,0
cell_001,Treatment,5
```

`condition` is optional; missing values are assigned to one group. Cell IDs must be unique within a condition. The public workflow accepts 5–1,000 cells per condition, integer counts from 0–100, and at least one positive count per condition.


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys


def find_orca_checkout():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "bayesorca").is_dir():
            return candidate
    return None


ORCA_ROOT = find_orca_checkout()
if ORCA_ROOT is not None:
    sys.path[:0] = [str(ORCA_ROOT), str(ORCA_ROOT / "src")]
elif importlib.util.find_spec("bayesorca") is None:
    if sys.version_info[:2] != (3, 12):
        raise RuntimeError("ORCA currently requires a Python 3.12 Colab runtime.")
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "git+https://github.com/sthsci/Orca.git@main",
        ]
    )

import bayesorca

print("bayesorca", bayesorca.__version__)
print("Python", sys.version.split()[0])


In [ ]:
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from bayesorca.event_counts import (
    MODEL_SPECS,
    InferenceSettings,
    build_condition_results_zip,
    evidence_table,
    normalize_condition_frame,
    run_condition_models,
    sample_count_frame,
    summary_table,
    validate_condition_frame,
)

USE_UPLOAD = False
OBSERVATION_TIME = 1.0
RUN_INFERENCE = False
MODEL_KEYS = list(MODEL_SPECS)


In [ ]:
def upload_one_csv():
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("Set USE_UPLOAD=True in Google Colab, or replace raw_data directly.") from exc
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one CSV file.")
    return pd.read_csv(BytesIO(next(iter(uploaded.values()))))


if USE_UPLOAD:
    raw_data = upload_one_csv()
else:
    control = sample_count_frame().assign(condition="Control")
    treatment = sample_count_frame().assign(
        condition="Treatment",
        count=lambda frame: frame["count"] + [1, 1, 0, 2, 1, 0, 2, 1, 1, 2, 0, 1],
    )
    raw_data = pd.concat([control, treatment], ignore_index=True).loc[
        :, ["cell_id", "condition", "count"]
    ]

mapped_data, mapping_message = normalize_condition_frame(raw_data, donor_aware=False)
data = validate_condition_frame(mapped_data, donor_aware=False)
print(mapping_message)
print(f"Validated {len(data):,} cells across {data['condition'].nunique()} condition(s).")
data.head()


In [ ]:
descriptive = (
    data.groupby("condition", as_index=False)
    .agg(
        cells=("cell_id", "size"),
        mean_count=("count", "mean"),
        variance=("count", "var"),
        zero_fraction=("count", lambda values: values.eq(0).mean()),
    )
)
display(descriptive.round(3))

axes = data.hist(
    column="count",
    by="condition",
    bins=range(int(data["count"].max()) + 2),
    figsize=(10, 4),
    sharex=True,
    sharey=True,
    color="#304B3D",
    rwidth=0.9,
)
plt.suptitle("Event-count distributions by condition")
plt.tight_layout()
plt.show()


## Fit and export

Each condition is fitted independently with the same model set and priors. The preview uses one chain and 128 SMC particles. For reported results, increase computation, repeat with multiple seeds/chains, inspect posterior stability, and justify the prior sensitivity of Bayes factors.


In [ ]:
if RUN_INFERENCE:
    settings = InferenceSettings(draws=128, chains=1, cores=1, seed=2026)
    results = run_condition_models(
        data,
        observation_time=OBSERVATION_TIME,
        settings=settings,
        model_keys=MODEL_KEYS,
        donor_aware=False,
    )

    evidence = pd.concat(
        [evidence_table(models).assign(condition=condition) for condition, models in results.items()],
        ignore_index=True,
    )
    posterior_summary = pd.concat(
        [summary_table(models).assign(condition=condition) for condition, models in results.items()],
        ignore_index=True,
    )
    display(evidence)
    display(posterior_summary)

    archive_path = Path("orca_event_count_analysis.zip")
    archive_path.write_bytes(
        build_condition_results_zip(
            results,
            data,
            OBSERVATION_TIME,
            settings,
            donor_aware=False,
        )
    )
    print("Saved", archive_path.resolve())
    try:
        from google.colab import files
        files.download(str(archive_path))
    except ImportError:
        pass
else:
    print("Data checks complete. Set RUN_INFERENCE = True when you are ready to fit.")


## Report with the result

Record the event definition, observation-time units, inclusion/exclusion rules, cell count per condition, model keys, prior settings, SMC particles/chains, random seed, ORCA version, and any sensitivity runs. A Bayes factor ranks only the models that were compared; it does not establish that the best candidate is biologically complete.
